# Архитектура, датасет...

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer
#bert-base-multilingual-cased
class ToxicBERTv2(nn.Module):
    def __init__(self, bert_model_name='DeepPavlov/rubert-base-cased', hidden_dim=512, dropout_prob=0.3):
        super(ToxicBERTv2, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name, output_hidden_states=True)
        h_size = self.bert.config.hidden_size

        for param in self.bert.parameters():
            param.requires_grad = False

        # --- Блок 1 ---
        # self.attn1 = nn.MultiheadAttention(embed_dim=h_size, num_heads=8, batch_first=True)
        # self.norm1_1 = nn.LayerNorm(h_size)
        # self.ffn1 = nn.Sequential(
        #     nn.Linear(h_size, h_size*2),
        #     nn.ReLU(),
        #     nn.Dropout(dropout_prob/2),
        #     nn.Linear(h_size*2, h_size),
        #     nn.ReLU(),
        #     nn.Dropout(dropout_prob/2)
        # )
        # self.norm1_2 = nn.LayerNorm(h_size)

        # # --- Блок 2 ---
        # self.attn2 = nn.MultiheadAttention(embed_dim=h_size, num_heads=8, batch_first=True)
        # self.norm2_1 = nn.LayerNorm(h_size)
        # self.ffn2 = nn.Sequential(
        #     nn.Linear(h_size, h_size*2),
        #     nn.ReLU(),
        #     nn.Dropout(dropout_prob/2),
        #     nn.Linear(h_size*2, h_size),
        #     nn.ReLU(),
        #     nn.Dropout(dropout_prob/2)
        # )
        # self.norm2_2 = nn.LayerNorm(h_size)

        input_head_size = h_size * 4 * 4

        self.head = nn.Sequential(
            nn.Linear(input_head_size, hidden_dim*2),
            nn.LayerNorm(hidden_dim*2),
            nn.ReLU(),
            nn.Dropout(dropout_prob/1.5),
            nn.Linear(hidden_dim*2, hidden_dim*3),
            nn.ReLU(),
            nn.Dropout(dropout_prob/1.5),
            nn.Linear(hidden_dim*3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(dropout_prob/2),
            nn.Linear(hidden_dim//2, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # Берем последние 4 слоя из hidden_states (кортеж из 13 слоев: 0 - embedding, 1-12 - layers)
        hidden_states = outputs.hidden_states[-4:]

        pooled_layers = []

        for layer in hidden_states:
            # layer shape: [Batch, Seq, Hidden]

            # 1. Среднее значение
            mean_p = torch.mean(layer, dim=1)
            # 2. Максимальное
            max_p, _ = torch.max(layer, dim=1)
            # 3. Минимальное
            min_p, _ = torch.min(layer, dim=1)
            # 4. Дисперсия (Variance)
            var_p = torch.var(layer, dim=1)

            # Склеиваем статистики одного слоя: [Batch, Hidden * 4]
            pooled_layers.append(torch.cat((mean_p, max_p, min_p, var_p), dim=1))

        # Склеиваем все 4 слоя вместе: [Batch, Hidden * 16]
        full_features = torch.cat(pooled_layers, dim=1)

        return self.head(full_features)

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer

class ToxicBERTv3(nn.Module):
    def __init__(self, bert_model_name='bert-base-multilingual-cased', hidden_dim=512, dropout_prob=0.3):
        super(ToxicBERTv2, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        h_size = self.bert.config.hidden_size


        for param in self.bert.parameters():
            param.requires_grad = False

        # --- Блок 1 ---
        self.attn1 = nn.MultiheadAttention(embed_dim=h_size, num_heads=8, batch_first=True)
        self.norm1_1 = nn.LayerNorm(h_size)
        self.ffn1 = nn.Sequential(
            nn.Linear(h_size, h_size*2),
            nn.ReLU(),
            nn.Dropout(dropout_prob/2),
            nn.Linear(h_size*2, h_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob/2)
        )
        self.norm1_2 = nn.LayerNorm(h_size)

        # --- Блок 2 ---
        self.attn2 = nn.MultiheadAttention(embed_dim=h_size, num_heads=8, batch_first=True)
        self.norm2_1 = nn.LayerNorm(h_size)
        self.ffn2 = nn.Sequential(
            nn.Linear(h_size, h_size*2),
            nn.ReLU(),
            nn.Dropout(dropout_prob/2),
            nn.Linear(h_size*2, h_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob/2)
        )
        self.norm2_2 = nn.LayerNorm(h_size)

        # --- Финальный классификатор ---
        # Вход в 3 раза больше из-за конкатенации (Mean + Max + Min)
        self.classifier = nn.Sequential(
            nn.Linear(h_size * 3, hidden_dim*2),
            nn.ReLU(),
            nn.Dropout(dropout_prob/1.5),
            nn.Linear(hidden_dim*2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob/1.5),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, input_ids, attention_mask):
        # 1. BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state # [B, L, H]

        # 2. Блок внимания 1 (Add & Norm)
        attn_out, _ = self.attn1(x, x, x)
        x = self.norm1_1(x + attn_out) # Residual + Norm
        ffn_out = self.ffn1(x)
        x = self.norm1_2(x + ffn_out)  # Residual + Norm

        # 3. Блок внимания 2 (Add & Norm)
        attn_out, _ = self.attn2(x, x, x)
        x = self.norm2_1(x + attn_out)
        ffn_out = self.ffn2(x)
        x = self.norm2_2(x + ffn_out)

        # 4. Triple Pooling (Mean, Max, Min) по размерности L (dim=1)
        mean_p = torch.mean(x, dim=1)           # [B, H]
        max_p, _ = torch.max(x, dim=1)          # [B, H]
        min_p, _ = torch.min(x, dim=1)          # [B, H]

        # Склеиваем векторы в один большой "супер-вектор"
        combined_x = torch.cat((mean_p, max_p, min_p), dim=1) # [B, H*3]

        # 5. Классификация
        logits = self.classifier(combined_x)
        return logits

# Загрузка датасета

In [ ]:
import kagglehub

path1 = kagglehub.dataset_download("blackmoon/russian-language-toxic-comments")
path2 = kagglehub.dataset_download("alexandersemiletov/toxic-russian-comments")
path3 = kagglehub.dataset_download("miklgr500/jigsaw-train-multilingual-coments-google-api")

100%|██████████| 500M/500M [00:09<00:00, 56.9MB/s]

Extracting files...


In [ ]:
path3

'/root/.cache/kagglehub/datasets/miklgr500/jigsaw-train-multilingual-coments-google-api/versions/2'

In [ ]:
import pandas as pd
import re

def process1(filepath):
    # Допустим, здесь разделитель — точка с запятой
    df = pd.read_csv(filepath)
    return df

def process2(filepath):
    data = []
    labels_map = {'NORMAL': 0, 'INSULT': 1, 'THREAT':1, 'OBSCENITY':1}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Регулярное выражение:
            # Находим всё, что начинается с __label__ до первого пробела, остальное — текст
            match = re.match(r'^((?:__label__[\w,]+ ?)+)\s+(.*)$', line)

            if match:
                labels_raw, text = match.groups()

                # Очищаем метки от префикса __label__ и разделяем, если их несколько
                # Из "__label__INSULT,__label__THREAT" -> "INSULT,THREAT"
                labels = ",".join(re.findall(r'__label__(\w+)', labels_raw))

                data.append({
                    'comment':text,
                    'toxic':0 if labels == "NORMAL" else 1
                })
    # Создаем DataFrame
    df = pd.DataFrame(data)
    return df

def process3(filepath):
    # Допустим, здесь разделитель — точка с запятой
    df = pd.read_csv(filepath)
    df = df.rename(columns={'comment_text': 'comment'})
    return df[['comment','toxic']]



In [ ]:
data_files = [
    (path1 + '/labeled.csv', process1),
    (path2 + '/dataset.txt', process2),
    (path3 + '/jigsaw-toxic-comment-train-google-ru-cleaned.csv', process3),
]

all_dfs = []

for file, func in data_files:
    temp_df = func(file)
    all_dfs.append(temp_df)

# Объединяем все в один финальный датафрейм
final_df = pd.concat(all_dfs, ignore_index=True)

In [ ]:
final_df[448280:448300]

,comment,toxic
448280,":Согласен. Это было, конечно, не незначительно...",0.0
448281,"""\n :: Не начинать дебаты заново, но я предлаг...",0.0
448282,ВЫ МАССОВАЯ ЧАСТЬ КРЕПА LOLOLOL.,0.0
448283,== 26 декабря ==\n\n В Великобритании Закон...,0.0
448284,"* Я верю вашему слову, поскольку у меня есть е...",0.0
448285,Джейми Ли Джонс не является членом вооруженных...,0.0
448286,"""\n\n ::::Спасибо. """,0.0
448287,"""\n : Да. Что ж, если это должно быть явное """"...",0.0
448288,== Смерть ==\n\n Дик Уинтерс скончался:\n\n ht...,0.0
448289,"""\n\n == Где мясо на этом рынке? ==\n\n Эта ст...",0.0


In [ ]:
class ToxicityDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
            text = str(self.df.iloc[idx]['comment'])
            score = self.df.iloc[idx]['toxic'] # Число от 0 до 1 из Kaggle

            # Превращаем в дискретный класс
            # Класс 0: Токсичный (если > 0.5)
            # Класс 1: Не токсичный (если <= 0.5)
            label = 0 if score > 0.5 else 1

            encoding = self.tokenizer(
                text,
                add_special_tokens=True,
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            return {
                'input_ids': encoding['input_ids'].flatten(),
                'attention_mask': encoding['attention_mask'].flatten(),
                'label': torch.tensor(label, dtype=torch.long) # CrossEntropy ждет Long
            }


In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')
modelv2 = ToxicBERTv2().to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
from torch.utils.data import random_split

full_dataset = ToxicityDataset(
    df=final_df,
    tokenizer=tokenizer,
    max_len=128
)

# 1. Определяем размеры
train_size = int(0.95 * len(full_dataset))
test_size = len(full_dataset) - train_size

# 2. Рандомно разбиваем
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

print(f"Размер обучающей выборки: {len(train_dataset)}")
print(f"Размер тестовой выборки: {len(test_dataset)}")

Размер обучающей выборки: 461556
Размер тестовой выборки: 24293


In [52]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

# Обучение

v2

In [ ]:
import torch
from tqdm.auto import tqdm # Для полоски прогресса
from google.colab import drive
import os

drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/ai-project-models'
os.makedirs(save_path, exist_ok=True)

def train_model(model, train_loader, val_loader, optimizer, criterion, device,scheduler, epochs=3):
    model.to(device)

    for epoch in range(epochs):
        # --- ФАЗА ОБУЧЕНИЯ ---
        model.train()
        train_loss = 0

        # Обертка для красивого вывода в консоль
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

        for batch in train_loop:
            # Перенос данных на GPU/CPU
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device) # Приводим к виду [batch, 1]

            # Обнуляем градиенты
            optimizer.zero_grad()

            # Forward pass (Прямой проход)
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

            # Backward pass (Обратный проход)
            loss.backward()

            # Обновление весов
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()
            train_loop.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss,
        }

        model_name = "toxic-bert-v3-epoch-" + str(epoch) + ".pth"
        full_path = os.path.join(save_path, model_name)
        torch.save(checkpoint, full_path)
        print(f"Модель сохранена по адресу: {full_path}")

        # --- ФАЗА ВАЛИДАЦИИ ---
        model.eval()
        val_loss = 0
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)

        with torch.no_grad(): # Отключаем расчет градиентов для экономии памяти
            for batch in val_loop:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        print(f"\n=> Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n")

    print("Обучение завершено!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## последняя модель

In [ ]:
from transformers import get_cosine_schedule_with_warmup
# 1. Модель и оптимизатор
# Для BERT лучше использовать AdamW и маленький learning rate
optimizer = torch.optim.AdamW(modelv2.parameters(), lr=4.6e-5)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=1430,
    num_training_steps=14000*4
)

# 2. Функция потерь
# Для токсичности 0...1 подойдет MSE или HuberLoss (если есть выбросы)
weights = torch.tensor([1.0, 9.0]).to(device) # Для классов [Токсично, Чисто]
criterion = torch.nn.CrossEntropyLoss(weight=weights)

# 3. Устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 4. Запуск
# Предполагается, что train_loader и val_loader уже созданы
train_model(
    model=modelv2,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scheduler=scheduler,
    epochs=1
)

Epoch 1/1 [Train]:   0%|          | 0/6984 [00:00<?, ?it/s]

Модель сохранена по адресу: /content/drive/MyDrive/ai-project-models/toxic-bert-v3-epoch-0.pth


Epoch 1/1 [Val]:   0%|          | 0/776 [00:00<?, ?it/s]


=> Epoch 1: Train Loss: 0.1458 | Val Loss: 0.1258

Обучение завершено!


In [ ]:
train_model(
    model=modelv2,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scheduler=scheduler,
    epochs=1
)

Epoch 1/1 [Train]:   0%|          | 0/6984 [00:00<?, ?it/s]

Модель сохранена по адресу: /content/drive/MyDrive/ai-project-models/toxic-bert-v3-epoch-0.pth


Epoch 1/1 [Val]:   0%|          | 0/776 [00:00<?, ?it/s]


=> Epoch 1: Train Loss: 0.1299 | Val Loss: 0.1172

Обучение завершено!


In [ ]:
train_model(
    model=modelv2,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    scheduler=scheduler,
    epochs=1
)

Epoch 1/1 [Train]:   0%|          | 0/13967 [00:00<?, ?it/s]

Модель сохранена по адресу: /content/drive/MyDrive/ai-project-models/toxic-bert-v2-epoch-0.pth


Epoch 1/1 [Val]:   0%|          | 0/1552 [00:00<?, ?it/s]


=> Epoch 1: Train Loss: 0.1038 | Val Loss: 0.0994

Обучение завершено!


In [ ]:
def predict_toxicity(text, model, tokenizer, device, max_len=128):
    # 1. Переводим модель в режим оценки (выключает Dropout и BatchNormalization)
    model.eval()
    model.to(device)

    # 2. Подготовка текста (токенизация)
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    # 3. Предсказание без расчета градиентов
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)

        # Если в конце модели нет Sigmoid, а задача 0-1,
        # можно применить его здесь для наглядности


    return outputs




In [ ]:
#modelv2.load_state_dict(torch.load('/content/drive/MyDrive/ai-project-models/toxic-bert-v2-epoch-3.pth'))
#modelv2.eval()

In [ ]:
my_text = "завались уже, достал!"

# Запускаем проверку
score = predict_toxicity(my_text, modelv2, tokenizer, device)

print(f"Текст: {my_text}")
print(score)


Текст: убить
tensor([[-4.8846,  3.3657]], device='cuda:0')
